# 8 — Paper Figures

Regenerates every figure in the paper, byte-for-byte identical to the versions
in `paper/figures/`. Each cell writes one PDF (for LaTeX) and one PNG (for
on-screen checking).

**Inputs, both opened read-only:**
- `./results/` — this project
- `../SEP_DataAugmentation/results/` — the companion study, used only for the
  normalization sweep and the base-learner and augmentation comparisons

**Output:** `./paper/figures/*.pdf` and `*.png`

| Figure | File |
|---|---|
| 1 | `dataset_overview.pdf` |
| 2 | `pipeline.pdf` |
| 3 | `related_work.pdf` |
| 4 | `preprocessing.pdf` |
| 5 | `design_choices.pdf` |
| 6 | `ensemble.pdf` |

Figures 1 and 2 need the raw per-channel CSVs and are placed last, since they
are the slowest to build.

**Run after:** notebooks 1–7.

## 1. Shared Style

In [1]:
"""Result figures for the SEP paper, built from both projects' result files.

Reads only the consolidated CSVs in the scratchpad; writes PDFs/PNGs into
the paper's figures/ directory. Never touches either results/ directory.
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

P1  = "../SEP_DataAugmentation/results"   # companion study, read only
P2  = "./results"                          # this project, read only
FIG = "./paper/figures"
os.makedirs(FIG, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.4, "ytick.major.size": 2.4,
    "xtick.labelsize": 6.4, "ytick.labelsize": 6.4,
    "axes.labelsize": 7.0, "axes.titlesize": 7.4, "legend.fontsize": 6.2,
    "axes.edgecolor": "0.35", "text.color": "0.0",
    "axes.labelcolor": "0.0", "xtick.color": "0.25", "ytick.color": "0.25",
})
BLUE, VERM, TEAL, GOLD = "#0072B2", "#D55E00", "#009E73", "#CC9A00"
GREY, LG = "0.45", "#DCDCDC"


def finish(ax, ylab=None, xlab=None, grid="y"):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.grid(True, axis=grid, color="0.88", lw=0.5, zorder=0)
    ax.set_axisbelow(True)
    if ylab:
        ax.set_ylabel(ylab)
    if xlab:
        ax.set_xlabel(xlab)

## 2. Load Results

In [2]:
# ══════════════════════════════════════════════════════════════
# Result loaders. Both results/ directories are opened read-only;
# nothing in either of them is written or moved.
#
# The companion study's files are headerless with 15 columns:
#   TP,TN,FP,FN,tss,hss1,hss2,gss,recall,f1,accuracy,
#   train_time,infer_time,far,bias
# They carry no Brier and no precision column, so precision is
# recovered from TP and FP.
# ══════════════════════════════════════════════════════════════
P1COLS = ["TP", "TN", "FP", "FN", "tss", "hss1", "hss2", "gss", "recall",
          "f1", "accuracy", "train_time", "infer_time", "far", "bias"]

CATS = {
    "Normalization": [("Raw", "nonorm"), ("Z-score", "zscore_all"),
                      ("Min-max", "minmax_all"), ("Log", "log_all"),
                      ("Hybrid", "hybrid")],
    "Borderline cleaning": [("Tomek Links", "hybrid_tomek")],
    "Oversampling": [("ADASYN 500", "tomek_rus500_adasyn500"),
                     ("SMOTE 500", "tomek_rus500_smote500"),
                     ("TimeGAN 500", "timegan_500"),
                     ("Diffusion 500", "diffusion_500")],
}
CLFS = {"svm": "SVM", "gru": "GRU", "patchtst": "PatchTST",
        "inceptiontime": "InceptionTime"}

rows = []
for cat, methods in CATS.items():
    for label, key in methods:
        for clf, clf_lab in CLFS.items():
            fp = os.path.join(P1, f"{clf}_{key}.txt")
            if not os.path.exists(fp):
                continue
            for run, line in enumerate(open(fp), start=1):
                line = line.strip()
                if not line:
                    continue
                parts = line.split(",")
                if len(parts) < 13:
                    continue
                d = dict(zip(P1COLS, [float(v) for v in parts] + [np.nan] * 2))
                d["precision"] = (d["TP"] / (d["TP"] + d["FP"])
                                  if d["TP"] + d["FP"] else 0.0)
                d["pod"] = d["recall"]
                d.update(category=cat, method=label, classifier=clf_lab, run=run)
                rows.append(d)

p1 = pd.DataFrame(rows)

sfs  = pd.read_csv(f"{P2}/gru_feature_selection_sfs.txt")
cmp_ = pd.read_csv(f"{P2}/gru_comparison.txt")
wmem = pd.read_csv(f"{P2}/ensemble_gru_adasyn_members.txt")
hyb  = pd.read_csv(f"{P2}/gru_hybridnorm.txt")
bens = pd.read_csv(f"{P2}/gru_baseline_ensemble.txt")
wfin = pd.read_csv(f"{P2}/ensemble_gru_adasyn_final.txt")
arch = pd.read_csv(f"{P2}/architecture_comparison.txt")
abl  = pd.read_csv(f"{P2}/ensemble_ablation.txt")
abl["variant"] = abl["variant"].str.strip('"')

print(f"companion study : {len(p1)} runs, {p1.classifier.nunique()} classifiers")
print(f"this project    : sfs {len(sfs)}, comparison {len(cmp_)}, "
      f"members {len(wmem)}")

companion study : 80 runs, 4 classifiers
this project    : sfs 275, comparison 50, members 48


## 3. Figure 3 — Comparison with Published Work

In [11]:
# FIG 3 -- comparison against published SEP forecasting models
# every "ours" row is read from the result files so it cannot drift
cal = arch[arch.kind == "calibrated"]
OURS = [
    ("Winnowed Ensemble",  wfin.tss.mean()),
    ("Baseline ensemble",  bens[bens.kind == "calibrated"].tss.mean()),
    ("Single GRU",         cmp_[(cmp_.kind == "raw") & (cmp_.dataset == "siamese")].tss.mean()),
    ("PatchTST",           cal[cal.model == "PatchTST"].tss.mean()),
    ("InceptionTime",      cal[cal.model == "InceptionTime"].tss.mean()),
    ("SVM (catch22)",      cal[cal.model == "SVM"].tss.mean()),
]
# TSS and the test-set class balance as reported by each study
PUBLISHED = [
    ("MEMPSEP-I [MEMPSEP1]", 0.63, "2:1"),
    ("UDM [UDM]", 0.59, "1:1"),
    ("Bi-LSTM [BiLSTMSEP]", 0.53, "42:1"),
    ("SMARP SVM, [SMARPSHARP]", 0.47, "1:1"),
    ("SEPNET-TS [SEPNET]", 0.45, "3:1"),
]
# barh draws upward, so published work goes in first and ours ends up on top
LIT = ([(n, v, r, False) for n, v, r in sorted(PUBLISHED, key=lambda t: t[1])] +
       [(n, v, "103:1", True) for n, v in sorted(OURS, key=lambda t: t[1])])
fig, ax = plt.subplots(figsize=(3.45, 2.75))
lab = [l.split(" [")[0] for l, *_ in LIT]
val = [v for _, v, *_ in LIT]
imb = [i for _, _, i, _ in LIT]
ours = [o for *_, o in LIT]
y = np.arange(len(LIT))
ax.barh(y, val, height=0.62,
        color=[TEAL if o else LG for o in ours],
        edgecolor=["0.35" if o else "0.72" for o in ours], lw=0.6, zorder=3)
for yi, v, im, o in zip(y, val, imb, ours):
    ax.text(v + 0.012, yi, f"{v:.2f}", va="center", ha="left", fontsize=6.2,
            color="0.0" if o else "0.35", zorder=4)
    ax.text(0.012, yi, im, va="center", ha="left", fontsize=5.2,
            color="white" if o else "0.45", zorder=4)
ax.set_yticks(y)
ax.set_yticklabels(lab, fontsize=6.2)
ax.set_xlim(0, 0.80)
ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8])
finish(ax, xlab="Test TSS", grid="x")
ax.text(0.012, len(LIT) - 0.42, "NonSEP : SEP in test set", fontsize=5.0,
        color="0.45", style="italic")
fig.tight_layout(pad=0.35)
fig.savefig(f"{FIG}/related_work.pdf")
fig.savefig(f"{FIG}/related_work.png", dpi=340)
plt.close(fig)
print("fig3 related_work OK")

# =====================================================================

fig3 related_work OK


## 4. Figure 4 — Preprocessing Evidence

In [7]:
# FIG 4 -- preprocessing evidence: normalization / channels / cleaning
fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.45),
                         gridspec_kw=dict(width_ratios=[1.0, 1.2, 1.05]))

ax = axes[0]
norm_order = ["Raw", "Z-score", "Min-max", "Log", "Hybrid"]
nsub = p1[(p1.category == "Normalization") & (p1.classifier == "GRU")]
for i, m in enumerate(norm_order):
    # the hybrid-normalized GRU is this project's own run; the other four
    # schemes come from the companion study, which is the only place they
    # were measured
    if m == "Hybrid":
        v = hyb[hyb.kind == "raw"].tss.values
    else:
        v = nsub[nsub.method == m].tss.values
    c = TEAL if m == "Hybrid" else "0.62"
    ax.bar(i, v.mean(), 0.60, color=c, edgecolor="none", zorder=3)
    ax.scatter(np.full_like(v, i, dtype=float), v, s=5, color="0.15", lw=0,
               zorder=5)
    off = 0.03 if v.mean() >= 0 else -0.09
    if m == "Log":            # sits just above its bar
        off += 0.012
    elif m == "Hybrid":       # lifted clear of the Log label beside it
        off += 0.085
    ax.text(i, v.mean() + off, f"{v.mean():.2f}", ha="center", fontsize=5.9,
            color="0.0", zorder=5)
ax.axhline(0, color="0.4", lw=0.6, zorder=4)
ax.set_xticks(range(len(norm_order)))
ax.set_xticklabels(norm_order, rotation=25, ha="right", fontsize=6.0)
ax.set_ylim(-0.32, 0.80)
finish(ax, ylab="Test TSS")
ax.set_title("(a)  Normalization", loc="left", fontsize=7.0, pad=4)

ax = axes[1]
rounds = sorted(sfs["round"].unique())
per = sfs.groupby(["round", "features"]).tss.mean().reset_index()
lo = per.groupby("round").tss.min()
hi = per.groupby("round").tss.max()
best = per.loc[per.groupby("round").tss.idxmax()].set_index("round")
ax.fill_between(rounds, lo.values, hi.values, color=VERM, alpha=0.13, lw=0,
                zorder=2, label="candidates")
ax.plot(rounds, best.tss.values, "-o", color=VERM, lw=1.1, ms=2.8,
        mfc="white", mew=0.9, zorder=4, label="greedy winner")
kbest = int(best.tss.idxmax())
ax.scatter([kbest], [best.tss.max()], s=40, facecolor="none", edgecolor=TEAL,
           lw=1.2, zorder=5)
ax.annotate("P4, Xl, Vx", xy=(kbest, best.tss.max()),
            xytext=(kbest + 1.3, best.tss.max() - 0.048), fontsize=5.8,
            color=TEAL, ha="left",
            arrowprops=dict(arrowstyle="-", lw=0.6, color=TEAL))
ax.set_xticks([1, 2, 3, 4, 6, 8, 10])
ax.set_ylim(0.55, 0.71)
finish(ax, ylab="Validation TSS", xlab="channels selected")
ax.set_title("(b)  Channel search", loc="left", fontsize=7.0, pad=4)
ax.legend(frameon=False, loc="lower right", handlelength=1.0,
          handletextpad=0.4, borderpad=0.1, labelspacing=0.25)

ax = axes[2]
names = {"baseline": "none", "deep_enn": "Deep ENN", "deep_tomek": "Deep Tomek",
         "deep_ae": "Autoencoder", "siamese": "Impostor\nTriplets"}
keys = ["baseline", "deep_enn", "deep_tomek", "deep_ae", "siamese"]
raw = cmp_[cmp_.kind == "raw"]
for i, k in enumerate(keys):
    v = raw[raw.dataset == k].tss.values
    c = TEAL if k == "siamese" else "0.55"
    ax.scatter(np.full_like(v, i, dtype=float) + np.linspace(-.09, .09, len(v)),
               v, s=6, color=c, alpha=0.75, lw=0, zorder=3)
    ax.plot([i - .22, i + .22], [v.mean()] * 2, color=c, lw=1.6, zorder=4)
ax.set_xticks(range(len(keys)))
ax.set_xticklabels([names[k] for k in keys], rotation=25, ha="right",
                   fontsize=5.9)
ax.set_ylim(0.60, 0.73)
finish(ax, ylab="TestTSS")
ax.set_title("(c)  Borderline cleaning", loc="left", fontsize=7.0, pad=4)

fig.subplots_adjust(left=0.075, right=0.995, top=0.905, bottom=0.265,
                    wspace=0.30)
fig.savefig(f"{FIG}/preprocessing.pdf")
fig.savefig(f"{FIG}/preprocessing.png", dpi=340)
plt.close(fig)
print("fig4 preprocessing OK (3 panels)")

# =====================================================================

fig4 preprocessing OK (3 panels)


## 5. Figure 5 — Design Choices

In [5]:
# FIG 5 -- design choices: base learner and augmentation method
fig, axes = plt.subplots(1, 2, figsize=(3.45, 2.28))

ax = axes[0]
clfs = ["GRU", "PatchTST", "SVM", "InceptionTime"]
csub = p1[(p1.category == "Normalization") & (p1.method == "Hybrid")]
for i, c in enumerate(clfs):
    v = csub[csub.classifier == c].tss.values
    col = TEAL if c == "GRU" else "0.62"
    ax.bar(i, v.mean(), 0.60, color=col, edgecolor="none", zorder=3)
    ax.scatter(np.full_like(v, i, dtype=float), v, s=5, color="0.15", lw=0,
               zorder=5)
    ax.text(i, v.mean() + 0.025, f"{v.mean():.2f}", ha="center", fontsize=5.8,
            color="0.0", zorder=5)
ax.set_xticks(range(len(clfs)))
ax.set_xticklabels(clfs, rotation=32, ha="right", fontsize=5.9)
ax.set_ylim(0, 0.80)
finish(ax, ylab="TSS")
ax.set_title("(a)  Base learner", loc="left", fontsize=7.0, pad=4)

ax = axes[1]
aug = ["ADASYN", "SMOTE", "TimeGAN", "Diffusion"]
ov = p1[(p1.category == "Oversampling") & (p1.classifier == "GRU")]
base = p1[(p1.category == "Borderline cleaning") &
          (p1.method == "Tomek Links") & (p1.classifier == "GRU")].tss.mean()
for i, m in enumerate(aug):
    v = ov[ov.method == f"{m} 500"].tss.values
    col = TEAL if m == "ADASYN" else "0.62"
    ax.bar(i, v.mean(), 0.60, color=col, edgecolor="none", zorder=3)
    ax.scatter(np.full_like(v, i, dtype=float), v, s=5, color="0.15", lw=0,
               zorder=5)
    ax.text(i, v.mean() + 0.025, f"{v.mean():.2f}", ha="center", fontsize=5.8,
            color="0.0", zorder=5)
ax.set_xticks(range(len(aug)))
ax.set_xticklabels(aug, rotation=32, ha="right", fontsize=5.9)
ax.set_ylim(0, 0.80)
finish(ax, ylab="TSS")
ax.set_title("(b)  Augmentation", loc="left", fontsize=7.0, pad=4)

fig.subplots_adjust(left=0.115, right=0.985, top=0.90, bottom=0.265,
                    wspace=0.40)
fig.savefig(f"{FIG}/design_choices.pdf")
fig.savefig(f"{FIG}/design_choices.png", dpi=340)
plt.close(fig)
print("fig5 design_choices OK")

# =====================================================================

fig5 design_choices OK


## 6. Figure 6 — Ensemble Analysis

In [8]:
# FIG 6 -- ensemble: stage progression and member pruning

fig, axes = plt.subplots(1, 2, figsize=(7.16, 2.30),
                         gridspec_kw=dict(width_ratios=[1.0, 1.15]))

ax = axes[0]
stages = [
    ("Hybrid\nNormalization", hyb[hyb.kind == "raw"].tss.mean(),
     hyb[hyb.kind == "raw"].brier.mean()),
    ("$+$ channel\nsearch", raw[raw.dataset == "baseline"].tss.mean(),
     raw[raw.dataset == "baseline"].brier.mean()),
    ("$+$ Impostor\nTriplets", raw[raw.dataset == "siamese"].tss.mean(),
     raw[raw.dataset == "siamese"].brier.mean()),
    ("Winnowed\nEnsemble", wfin.tss.mean(), wfin.brier.mean()),
]
xs = np.arange(len(stages))
t = [v[1] for v in stages]
b = [v[2] for v in stages]
ax.plot(xs, t, "-o", color=TEAL, lw=1.3, ms=3.6, mfc="white", mew=1.0,
        zorder=4, label="TSS")
for xi, v in zip(xs, t):
    ax.text(xi, v + 0.006, f"{v:.3f}", ha="center", fontsize=5.7, color=TEAL)
ax.set_xticks(xs)
ax.set_xticklabels([v[0] for v in stages], fontsize=5.7)
lo_t, hi_t = min(t), max(t)
ax.set_ylim(lo_t - 0.02, hi_t + 0.02)
finish(ax, ylab="Test TSS")
ax2 = ax.twinx()
ax2.plot(xs, b, "-s", color=VERM, lw=1.0, ms=3.0, mfc="white", mew=0.9,
         zorder=4, label="Brier")
ax2.set_ylabel("Brier score", color=VERM, fontsize=7.0, labelpad=1.5)
ax2.tick_params(axis="y", colors=VERM, labelsize=6.4)
lo_b, hi_b = min(b), max(b)
ax2.set_ylim(lo_b - 0.008, hi_b + 0.008)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_color(VERM)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, frameon=False, loc="lower right", handlelength=1.2,
          handletextpad=0.4, borderpad=0.1, labelspacing=0.25)
ax.set_title("(a)  Pipeline stages", loc="left", fontsize=7.0, pad=4)

ax = axes[1]
# rank on the score the ensemble actually selected with; older result
# files predate the rank_score column, so fall back to the pooled score
rank_col = "rank_score" if "rank_score" in wmem.columns else "combined_train_val_tss"
w2 = wmem.sort_values(rank_col, ascending=False).reset_index()
n_kept = int(w2.selected.sum())
xs = np.arange(len(w2))
ax.bar(xs, w2.tss.values, 0.80,
       color=[TEAL if s_ else LG for s_ in w2.selected], edgecolor="none",
       zorder=3)
ax.axhline(w2[w2.selected].tss.mean(), color=TEAL, lw=0.9, ls=(0, (3, 2)),
           zorder=5)
ax.axhline(w2[~w2.selected].tss.mean(), color="0.45", lw=0.9, ls=(0, (1, 1.6)),
           zorder=5)
ax.set_xlim(-1, len(w2))
ax.set_ylim(w2.tss.min() - 0.02, w2.tss.max() + 0.03)
ax.set_xticks([0, 9, 19, 29, 39, 47])
ax.set_xticklabels(["1", "10", "20", "30", "40", "48"])
finish(ax, ylab="Test TSS", xlab="member, ranked by selection score")
ax.legend(handles=[Patch(facecolor=TEAL, label=f"kept (top {n_kept})"),
                   Patch(facecolor=LG, label="discarded")],
          frameon=False, loc="upper right", handlelength=0.9,
          handletextpad=0.4, borderpad=0.1, labelspacing=0.25)
ax.set_title("(b)  Member pruning", loc="left", fontsize=7.0, pad=4)

fig.subplots_adjust(left=0.075, right=0.985, top=0.905, bottom=0.215,
                    wspace=0.46)
fig.savefig(f"{FIG}/ensemble.pdf")
fig.savefig(f"{FIG}/ensemble.png", dpi=340)
plt.close(fig)
print("fig6 ensemble OK")

fig6 ensemble OK


## 7. Figure 1 — Dataset Construction

Reads the ten raw per-channel CSVs in `./dataset/`. Only the rows for the
plotted event are loaded, so this stays fast despite the 697 MB source.

In [7]:
"""Builds Fig. 1 (dataset construction) for the SEP paper.

Real traces from event #9391 (gt100) of the source catalog, 24 h of
pre-onset telemetry at 5-min cadence.  Output is vector PDF for LaTeX
plus a PNG for on-screen checking.
"""
import pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

OUT = "./paper/figures/dataset_overview"
EVENT = 9391          # a >100 MeV SEP event from the catalog
DATASET_DIR = "./dataset"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "xtick.major.size": 2,
})

OMNI = ["V", "Vx", "Np", "Tp", "F"]
GOES = ["Xl", "Xs", "P4", "P5", "P6"]
LOG = {"P4", "P5", "P6", "Xl", "Xs"}
PRETTY = {"V": "$V$", "Vx": "$V_x$", "Np": "$N_p$", "Tp": "$T_p$", "F": "$F$",
          "Xl": "Xl", "Xs": "Xs", "P4": "P4", "P5": "P5", "P6": "P6"}
C_OMNI, C_GOES = "#0072B2", "#D55E00"

# pull just this event's row from each per-channel CSV
import pandas as pd
FEATS = ["F", "Np", "P4", "P5", "P6", "Tp", "V", "Vx", "Xl", "Xs"]
series = {}
for f in FEATS:
    df = pd.read_csv(f"{DATASET_DIR}/{f}.csv",
                     skiprows=lambda i: i > 0 and i != EVENT + 1,
                     low_memory=False)
    series[f] = df.iloc[0, -288:].apply(pd.to_numeric,
                                        errors="coerce").values.astype(float)
d = {"X": np.stack([series[f] for f in FEATS], axis=-1)[None, ...]}
row = 0
fidx = {f: i for i, f in enumerate(FEATS)}
t = np.arange(288) * 5 / 60 - 24

fig = plt.figure(figsize=(3.45, 4.05))
gs = GridSpec(10, 1, figure=fig, hspace=0.0,
              left=0.175, right=0.97, top=0.935, bottom=0.215)

order = OMNI + GOES
axes = []
for k, f in enumerate(order):
    ax = fig.add_subplot(gs[k, 0])
    v = d["X"][row, :, fidx[f]].astype(float)
    color = C_OMNI if f in OMNI else C_GOES
    if f in LOG:
        ax.set_yscale("log")
        v = np.where(v > 0, v, np.nan)
    ax.plot(t, v, lw=0.6, color=color, solid_joinstyle="round")
    ax.set_xlim(-24, 0)
    ax.set_yticks([])
    ax.yaxis.set_minor_locator(matplotlib.ticker.NullLocator())
    ax.tick_params(axis="y", which="both", left=False, right=False, length=0,
                   labelleft=False)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.4)
    ax.spines["bottom"].set_color("0.78")
    ax.text(-0.028, 0.5, PRETTY[f], transform=ax.transAxes, ha="right",
            va="center", fontsize=7.2, color=color)
    ax.axvline(0, color="0.35", lw=0.7, ls=(0, (2.2, 1.6)), zorder=0)
    if k < 9:
        ax.set_xticklabels([])
        ax.tick_params(axis="x", length=0)
    axes.append(ax)

ax_last = axes[-1]
ax_last.set_xticks([-24, -18, -12, -6, 0])
ax_last.set_xticklabels(["$-24$", "$-18$", "$-12$", "$-6$", "0"], fontsize=6.6)
ax_last.spines["bottom"].set_color("0.3")
ax_last.spines["bottom"].set_linewidth(0.5)
ax_last.set_xlabel("hours before flare onset   (288 steps, 5-min cadence)",
                   fontsize=6.9, labelpad=2.0)

axes[0].text(-0.4, 1.10, "flare onset",
             transform=axes[0].get_xaxis_transform(),
             ha="right", va="bottom", fontsize=6.6, color="0.35")
axes[0].annotate("", xy=(0, 1.13), xytext=(-0.35, 1.13),
                 xycoords=axes[0].get_xaxis_transform(),
                 arrowprops=dict(arrowstyle="-|>", lw=0.6, color="0.35",
                                 shrinkA=0, shrinkB=0))

# --- source brackets -------------------------------------------------
def bracket(ax_top, ax_bot, label, color):
    x = 0.048
    y0 = ax_bot.get_position().y0
    y1 = ax_top.get_position().y1
    fig.add_artist(Line2D([x, x], [y0, y1], color=color, lw=0.9))
    for y in (y0, y1):
        fig.add_artist(Line2D([x, x + 0.015], [y, y], color=color, lw=0.9))
    fig.text(x - 0.014, (y0 + y1) / 2, label, rotation=90, ha="center",
             va="center", fontsize=7.4, color=color)

fig.canvas.draw()
bracket(axes[0], axes[4], "OMNI", C_OMNI)
bracket(axes[5], axes[9], "GOES", C_GOES)

# --- summary band ----------------------------------------------------
axb = fig.add_axes([0.055, 0.018, 0.915, 0.105])
axb.set_xlim(0, 1)
axb.set_ylim(0, 1)
axb.set_xticks([])
axb.set_yticks([])
for s in axb.spines.values():
    s.set_linewidth(0.5)
    s.set_color("0.80")
axb.set_facecolor("none")

axb.text(0.5, 0.68,
         r"stack 10 channels  $\Rightarrow$  "
         r"$\mathbf{X}\in\mathbb{R}^{17{,}794\times288\times10}$",
         ha="center", va="center", fontsize=7.4)
axb.text(0.5, 0.24,
         "labels:   17,625 NonSEP   and   169 SEP        (104 : 1)",
         ha="center", va="center", fontsize=6.9, color="0.0")

fig.savefig(OUT + ".pdf")
fig.savefig(OUT + ".png", dpi=400)
print("wrote", OUT + ".pdf / .png")

wrote ./paper/figures/dataset_overview.pdf / .png


## 8. Figure 2 — Pipeline and Architectures

A schematic; every number in it is quoted from the notebooks rather than
recomputed here.

In [8]:
"""Builds Fig. 2 (end-to-end pipeline + architectures) for the SEP paper.

Double-column IEEEtran figure*: \textwidth = 7.16 in.
Two stacked axes, data coords in inches (aspect='equal'):
  ax  -- stage flow          (7.16 x 2.62)
  axa -- model architectures (7.16 x 1.62)
Layer blocks mirror GRUModel / TripletEncoder / MetaVoter in the notebooks.
No abbreviations anywhere; a post-draw check reports text that overruns.
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle, FancyArrowPatch, Circle

OUT = "/Users/samskanderi/Documents/Claude/SEP_DataAugmentation 2/paper/figures/pipeline"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
})

BLUE, VERM, TEAL = "#0072B2", "#D55E00", "#009E73"
GREY, LGREY, EDGE = "0.0", "#D9D9D9", "0.62"
W, HTOP, HBOT = 7.16, 2.40, 1.14
H = HTOP + HBOT

fig = plt.figure(figsize=(W, H))
ax = fig.add_axes([0, HBOT / H, 1, HTOP / H])
ax.set_xlim(0, W); ax.set_ylim(0.12, 2.52); ax.set_aspect("equal")
ax.axis("off")
axa = fig.add_axes([0, 0, 1, HBOT / H])
axa.set_xlim(0, W); axa.set_ylim(0.12, 1.26); axa.set_aspect("equal")
axa.axis("off")

BY0, BY1, TBAR = 0.22, 2.42, 0.34
P = {"in": (0.03, 0.81), "s1": (0.94, 2.04), "s2": (2.17, 3.27),
     "s3": (3.40, 4.60), "s4": (4.73, 6.20), "out": (6.33, 7.13)}
checks = []


def panel(key, title, accent, two=True):
    x0, x1 = P[key]
    ax.add_patch(FancyBboxPatch((x0, BY0), x1 - x0, BY1 - BY0,
                                boxstyle="round,pad=0,rounding_size=0.05",
                                linewidth=0.7, edgecolor=EDGE,
                                facecolor="white", zorder=1))
    ax.add_patch(Rectangle((x0, BY1 - TBAR), x1 - x0, TBAR,
                           facecolor=accent, alpha=0.11, lw=0, zorder=1.5))
    t = ax.text((x0 + x1) / 2, BY1 - TBAR / 2, title, ha="center",
                va="center", fontsize=6.5, color=accent, zorder=3,
                linespacing=1.35)
    checks.append((t, key))
    return x0, x1


def put(key, y, txt, size=6.0, color=GREY):
    x0, x1 = P[key]
    t = ax.text((x0 + x1) / 2, y, txt, ha="center", va="center",
                fontsize=size, color=color, zorder=3)
    checks.append((t, key))


# ---------------------------------------------------------------- input
x0, x1 = panel("in", "Input", "0.15"); cx = (x0 + x1) / 2
for i, dx in enumerate([0.09, 0.045, 0.0]):
    ax.add_patch(Rectangle((cx - 0.235 + dx, 1.74 - dx * 0.9), 0.38, 0.26,
                           facecolor=["#EDEDED", "#E2E2E2", "white"][i],
                           edgecolor="0.55", lw=0.55, zorder=2 + i))
ax.text(cx - 0.045, 1.87, r"$288\!\times\!10$", ha="center", va="center",
        fontsize=5.8, zorder=6)
put("in", 1.50, "OMNI + GOES", 6.0)
put("in", 1.26, "17,794 events", 6.0)
put("in", 1.00, "chronological", 5.9)
put("in", 0.83, "70/10/20 split", 5.7)
put("in", 0.55, "NonSEP : SEP", 5.5, "0.0")
put("in", 0.38, "104 : 1", 6.6, VERM)

# ------------------------------------------------- I. hybrid normalization
x0, x1 = panel("s1", "I. Hybrid\nNormalization", BLUE); cx = (x0 + x1) / 2
hy = 1.74
for i, v in enumerate([0.24, 0.18, 0.10, 0.06, 0.04]):
    ax.add_patch(Rectangle((cx - 0.36 + i * 0.044, hy), 0.039, v,
                           facecolor=BLUE, alpha=0.5, lw=0, zorder=3))
for i, v in enumerate([0.05, 0.12, 0.24, 0.15, 0.06]):
    ax.add_patch(Rectangle((cx + 0.14 + i * 0.044, hy), 0.039, v,
                           facecolor=BLUE, alpha=0.85, lw=0, zorder=3))
ax.add_patch(FancyArrowPatch((cx - 0.10, hy + 0.10), (cx + 0.09, hy + 0.10),
                             arrowstyle="-|>", mutation_scale=6, lw=0.8,
                             color="0.45", zorder=3))
put("s1", 1.50, r"$\log_{10}$ then $z$-score", 6.0, BLUE)
put("s1", 1.32, "8 heavy-tailed channels", 5.4)
put("s1", 1.06, r"$z$-score only", 6.0, BLUE)
put("s1", 0.88, r"$V$ and $V_x$", 5.6)
put("s1", 0.58, "fitted on training", 5.9)
put("s1", 0.40, "data only", 5.9)

# ------------------------------------------------------ II. SFS selection
x0, x1 = panel("s2", "II. Channel\nSearch", VERM); cx = (x0 + x1) / 2
sel = {0: VERM, 5: VERM, 7: BLUE}
s, g = 0.115, 0.032
gx, gy = cx - (5 * s + 4 * g) / 2, 1.90
for i in range(10):
    r, c = divmod(i, 5)
    ax.add_patch(Rectangle((gx + c * (s + g), gy - r * (s + g)), s, s,
                           facecolor=sel.get(i, LGREY),
                           edgecolor="0.55" if i in sel else "0.82",
                           lw=0.5, zorder=3))
put("s2", 1.44, r"10 $\rightarrow$ 3 channels", 6.2, "0.0")
put("s2", 1.20, "P4,  Xl,  Vx", 6.4, VERM)
put("s2", 0.92, "55 subsets tested", 5.9)
put("s2", 0.68, "ranked by GRU TSS", 5.9)

# ------------------------------------------------- III. impostor triplets
x0, x1 = panel("s3", "III. Impostor\nTriplets", TEAL); cx = (x0 + x1) / 2
rng = np.random.default_rng(3)
ax.scatter(np.clip(rng.normal(cx - 0.21, 0.13, 42), x0 + 0.07, x1 - 0.07),
           rng.normal(1.86, 0.095, 42), s=2.0, c=LGREY, edgecolors="0.65",
           linewidths=0.22, zorder=3)
ax.scatter(rng.normal(cx + 0.21, 0.06, 14), rng.normal(1.86, 0.055, 14),
           s=3.0, c=VERM, lw=0, zorder=4)
ax.scatter(rng.normal(cx + 0.20, 0.055, 4), rng.normal(1.87, 0.045, 4),
           s=11, marker="x", c=TEAL, linewidths=0.85, zorder=5)
ax.add_patch(Circle((cx + 0.21, 1.865), 0.125, fill=False, ec=TEAL, lw=0.65,
                    ls=(0, (2, 1.6)), zorder=4))
ax.text(cx - 0.31, 1.66, "NonSEP", fontsize=5.4, color="0.0", ha="center",
        zorder=3)
ax.text(cx + 0.29, 1.66, "SEP", fontsize=5.4, color=VERM, ha="center",
        zorder=3)
put("s3", 1.44, r"embed with $g_\theta$", 6.0)
put("s3", 1.20, "out-of-fold, 5 folds", 5.9)
put("s3", 0.94, r"$\tau$ = 5th percentile", 5.9)
put("s3", 0.77, "of SEP-to-SEP distance", 5.5)
put("s3", 0.48, r"$\times$   75 impostors removed", 5.7, TEAL)

# --------------------------------------------------- IV. winnowed ensemble
x0, x1 = panel("s4", "IV. Winnowed\nEnsemble", TEAL); cx = (x0 + x1) / 2
keep = {2, 5, 9, 14, 22, 28, 38, 45}
ms, mg = 0.068, 0.018
mx = cx - (16 * ms + 15 * mg) / 2
my = 1.90
for i in range(48):
    r, c = divmod(i, 16)
    ax.add_patch(Rectangle((mx + c * (ms + mg), my - r * (ms + mg)), ms, ms,
                           facecolor=TEAL if i in keep else LGREY,
                           edgecolor="0.55" if i in keep else "0.83",
                           lw=0.42, zorder=3))
put("s4", 1.58, "48 GRU members", 6.0)
put("s4", 1.40, r"16 bootstrap draws $\times$ 3 settings", 5.5)
ax.add_patch(FancyArrowPatch((cx, 1.30), (cx, 1.18), arrowstyle="-|>",
                             mutation_scale=7, lw=0.9, color="0.40", zorder=4))
put("s4", 1.05, "prune to the best 8", 6.1, TEAL)
# eight bars whose heights are the rank weights, merging into one output
bw, bg = 0.052, 0.022
bx0 = cx - (8 * bw + 7 * bg) / 2 - 0.10
wts = np.linspace(1.0, 0.4, 8)
for i, wv in enumerate(wts):
    h = 0.10 + 0.16 * wv
    ax.add_patch(Rectangle((bx0 + i * (bw + bg), 0.62), bw, h,
                           facecolor=TEAL, alpha=0.35 + 0.5 * wv, lw=0, zorder=3))
bx1 = bx0 + 8 * bw + 7 * bg
ax.add_patch(FancyArrowPatch((bx1 + 0.02, 0.72), (bx1 + 0.16, 0.72),
                             arrowstyle="-|>", mutation_scale=6, lw=0.7,
                             color="0.40", zorder=4))
ax.add_patch(Circle((bx1 + 0.21, 0.72), 0.026, facecolor="0.25", lw=0, zorder=4))
ax.text(bx1 + 0.26, 0.72, r"$\hat{p}$", ha="left", va="center", fontsize=6.6,
        zorder=4)
put("s4", 0.44, "rank-weighted mean", 5.9)

# --------------------------------------------------------------- output
x0, x1 = panel("out", "Forecast", "0.15"); cx = (x0 + x1) / 2
ax.add_patch(FancyBboxPatch((cx - 0.33, 1.74), 0.66, 0.26,
                            boxstyle="round,pad=0,rounding_size=0.035",
                            lw=0.6, edgecolor=VERM, facecolor="white",
                            zorder=3))
ax.text(cx, 1.87, "threshold 0.54", ha="center", va="center", fontsize=5.6,
        color=VERM, zorder=4)
put("out", 1.54, "weighted mean", 5.7)
put("out", 1.38, "of 8 members", 5.7)
put("out", 1.10, r"$\hat{p}\in[0,1]$", 6.4, "0.0")
put("out", 0.84, "scored by", 5.6)
put("out", 0.66, "TSS", 5.9)
put("out", 0.50, "PoD", 5.9)
put("out", 0.34, "Brier score", 5.9)

# ------------------------------------------------------------- arrows
seq = ["in", "s1", "s2", "s3", "s4", "out"]
for a, b in zip(seq, seq[1:]):
    ax.add_patch(FancyArrowPatch((P[a][1] + 0.012, 1.30),
                                 (P[b][0] - 0.012, 1.30), arrowstyle="-|>",
                                 mutation_scale=8, lw=1.0, color="0.40",
                                 shrinkA=0, shrinkB=0, zorder=4))

# ==================================================================
#  architectures
# ==================================================================
axa.plot([0.03, 7.13], [1.20, 1.20], lw=0.7, color="0.80", zorder=1)
axa.text(0.03, 1.11, "Model architectures", fontsize=6.5, color="0.0",
         ha="left", va="center")
axa.text(7.13, 1.11, r"$H\in\{32,64,128\}$  and  $p\in\{0.1,0.3\}$ "
         "across the three member settings", fontsize=5.5, color="0.0",
         ha="right", va="center")

BW, GAP, BH, X0 = 0.62, 0.08, 0.30, 0.86

ENC = [r"$288\times3$" "\ninput", "GRU\n$H$ units", "last\ntime step",
       "dropout $p$", "linear 32", r"$\ell_2$" "\nnormalize"]
CLF = [r"$288\times3$" "\ninput", "GRU\n$H$ units", "last\ntime step",
       "batch\nnorm", "dropout $p$", "linear 32\nReLU", "dropout\n0.2",
       "linear 1"]

ROWS = [(0.82, r"$g_\theta$", "encoder", "stage III", TEAL, ENC,
         r"unit embedding $\mathbf{e}$"),
        (0.36, r"$f$", "classifier", "stages II, IV", BLUE, CLF,
         r"sigmoid $\rightarrow p$")]

for y, sym, name, stage, col, blocks, tail in ROWS:
    axa.text(0.05, y + 0.05, sym, fontsize=7.4, color=col, ha="left",
             va="center")
    axa.text(0.21, y + 0.055, name, fontsize=6.1, color="0.0", ha="left",
             va="center")
    axa.text(0.05, y - 0.09, stage, fontsize=5.4, color=col, ha="left",
             va="center")
    for i, b in enumerate(blocks):
        bx = X0 + i * (BW + GAP)
        io = i == 0
        axa.add_patch(FancyBboxPatch(
            (bx, y - BH / 2), BW, BH,
            boxstyle="round,pad=0,rounding_size=0.035",
            lw=0.6, edgecolor="0.72" if io else col,
            facecolor="#F2F2F2" if io else "white", zorder=3))
        axa.text(bx + BW / 2, y, b, fontsize=5.3,
                 color="0.0", ha="center", va="center",
                 zorder=4, linespacing=1.35)
        if i:
            axa.add_patch(FancyArrowPatch(
                (bx - GAP, y), (bx - 0.012, y), arrowstyle="-|>",
                mutation_scale=5, lw=0.6, color="0.55", shrinkA=0, shrinkB=0,
                zorder=3))
    ex = X0 + len(blocks) * (BW + GAP)
    axa.add_patch(FancyArrowPatch((ex - GAP, y), (ex - 0.012, y),
                                  arrowstyle="-|>", mutation_scale=5, lw=0.6,
                                  color="0.55", shrinkA=0, shrinkB=0, zorder=3))
    axa.text(ex + 0.03, y, tail, fontsize=5.9, color=col, ha="left",
             va="center", zorder=4)

# ------------------------------------------------------- overflow check
fig.canvas.draw()
r = fig.canvas.get_renderer()
inv = ax.transData.inverted()
bad = []
for t, key in checks:
    bb = t.get_window_extent(renderer=r).transformed(inv)
    x0, x1 = P[key]
    if bb.x0 < x0 + 0.025 or bb.x1 > x1 - 0.025:
        bad.append(f"  {key}: '{t.get_text()[:34]}'  "
                   f"[{bb.x0:.2f},{bb.x1:.2f}] vs [{x0:.2f},{x1:.2f}]")
inva = axa.transData.inverted()
for t in axa.texts:
    bb = t.get_window_extent(renderer=r).transformed(inva)
    if bb.x0 < 0.02 or bb.x1 > 7.14:
        bad.append(f"  arch: '{t.get_text()[:34]}' [{bb.x0:.2f},{bb.x1:.2f}]")
print("OVERFLOW:" if bad else "no overflow")
print("\n".join(bad))

fig.savefig(OUT + ".pdf")
fig.savefig(OUT + ".png", dpi=340)
print("wrote", OUT + ".pdf / .png")


no overflow

wrote ./paper/figures/pipeline.pdf / .png


wrote ./paper/figures/pipeline.pdf / .png


## 8. Figure 6 — Component Ablation of the Winnowed Ensemble


In [10]:
# FIG 6 -- component ablation of the Winnowed Ensemble
EVENT = 1 / 34.0
TEAL, VERM, GREY = "#009E73", "#D55E00", "0.55"
EVENT = 1 / 34.0

w = wfin
a = abl
full = w.tss.mean()

ROWS = [("No hyperparameter\ndiversity", "No hyperparameter diversity"),
        ("No bootstrap\nresampling",     "No bootstrap"),
        ("No synthetic\naugmentation",   "No synthetic augmentation"),
        ("Equal weights,\nno rank weighting", "Equal weights"),
        ("No pruning\n(all 48 members)", "No pruning (all 48)")]

XLAB = -0.0755          # fixed column for the absolute-TSS labels
fig, ax = plt.subplots(figsize=(3.45, 2.60))

ax.axvspan(-EVENT, EVENT, color=TEAL, alpha=0.11, lw=0, zorder=1)
ax.axvline(0, color=TEAL, lw=1.1, zorder=3)
for e in (-EVENT, EVENT):
    ax.axvline(e, color=TEAL, lw=0.6, ls=(0, (1.6, 1.6)), alpha=0.75, zorder=2)

for i, (lab, key) in enumerate(ROWS):
    v = a[a.variant == key].tss.values - full
    m = v.mean()
    ax.plot([0, m], [i, i], color="0.78", lw=0.9, zorder=2)
    ax.scatter(v, [i] * len(v), s=10, facecolor="white", edgecolor=GREY,
               lw=0.7, zorder=4)
    ax.scatter([m], [i], s=32, color=VERM, lw=0, zorder=5)
    ax.text(XLAB, i, f"{full + m:.3f}", ha="left", va="center",
            fontsize=6.0, color="0.30", zorder=6)

ax.set_yticks(range(len(ROWS)))
ax.set_yticklabels([l for l, _ in ROWS], fontsize=6.1, linespacing=1.2)
ax.set_ylim(-0.6, len(ROWS) - 0.15)
ax.set_xlim(-0.079, 0.038)
ax.set_xticks([-0.04, -0.02, 0.0])
ax.set_xticklabels(["$-0.04$", "$-0.02$", "0"])
ax.set_xlabel("change in Test TSS when the component is removed", labelpad=2.5)
for sp in ("top", "right", "left"):
    ax.spines[sp].set_visible(False)
ax.grid(True, axis="x", color="0.90", lw=0.5, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(axis="y", length=0)

ax.text(0.0035, len(ROWS) - 0.42, f"full ensemble\nTSS $=$ {full:.3f}",
        fontsize=6.0, color=TEAL, va="center", linespacing=1.25)
ax.text(-EVENT + 0.0015, -0.42, "band: $\\pm$ one test event",
        fontsize=5.6, color="0.42", style="italic", va="center", ha="left")
ax.text(XLAB, len(ROWS) - 0.42, "Test TSS", fontsize=6.0, color="0.30",
        ha="left", va="center")

fig.tight_layout(pad=0.35)
fig.savefig(f"{FIG}/ablation.pdf")
fig.savefig(f"{FIG}/ablation.png", dpi=340)
plt.close(fig)
print(f"full = {full:.4f}")
for lab, key in ROWS:
    v = a[a.variant == key].tss.values
    print(f"  {key:<30} {v.mean():.3f}  delta {v.mean()-full:+.3f}")


full = 0.7140
  No hyperparameter diversity    0.700  delta -0.014
  No bootstrap                   0.687  delta -0.027
  No synthetic augmentation      0.677  delta -0.037
  Equal weights                  0.667  delta -0.047
  No pruning (all 48)            0.658  delta -0.056
